# 📊 Business Metrics Mart

**Purpose**: Analytics Engineering layer providing curated business metrics for decision-making

**Architecture**: Gold Layer → Metrics Mart (Pre-computed Tables)

---

## 🎯 Architecture Decision: Tables vs Views vs Materialized Views

### Why Tables (Current Implementation)?

**Materialize Views** would be ideal BUT are **not enabled on Serverless compute** (requires preview feature enrollment). Given this constraint:

✅ **Tables** = Best choice for Serverless:
* Pre-computed aggregations (fast dashboard queries)
* Full control over refresh logic
* Can add custom DQ checks and logging
* Scheduled refresh via Databricks Jobs

❌ **Regular Views** would be too slow:
* Complex aggregations recompute on every query
* Poor dashboard performance
* Not suitable for BI tools

🔒 **Materialized Views** (future upgrade path):
* When your workspace enables MVs on Serverless, convert tables to MVs with `TRIGGER ON UPDATE`
* Automatic refresh when upstream data changes
* Zero maintenance with Predictive Optimization

---

## Metrics Categories

### 1️⃣ Holiday Sales Performance
- Revenue impact of holidays vs regular days
- Holiday lift analysis
- Pre/post holiday trends
- Holiday ranking by performance

### 2️⃣ Customer Behavior Analytics
- Customer Lifetime Value (CLV) by cohort
- Repeat purchase patterns
- Churn & retention analysis
- RFM segmentation

### 3️⃣ Product Performance
- Revenue & volume leaders
- Product affinity (basket analysis)
- Holiday vs regular performance
- Inventory velocity classification

### 4️⃣ Time-based Trends
- Day of week patterns
- Month-over-month growth
- Seasonal patterns
- Year-over-year comparisons

### 5️⃣ Operational KPIs
- Fulfillment performance
- On-time delivery rate
- Order backlog management

---

**Data Sources**:
- `workspace.gold.fact_sales_enriched` ✅ **Uses enriched table with holiday data**
- `workspace.gold.dim_customers`
- `workspace.gold.dim_products`

**Output**: Each metric category saved as `workspace.gold.metrics_*` table (refreshed via scheduled jobs)

## 1️⃣ Holiday Sales Performance Metrics

**Business Value**:
- Identify which holidays drive the most revenue
- Measure holiday lift to optimize marketing spend
- Understand customer behavior around holidays for inventory planning
- Compare pre/post holiday trends for promotional timing

**Metrics Included**:
- Holiday vs Non-Holiday Revenue & Volume
- Holiday Lift Percentage
- Pre-Holiday (7 days before) vs Holiday vs Post-Holiday (7 days after)
- Holiday Rankings by Revenue & AOV

In [0]:
%sql
-- Create comprehensive holiday sales performance metrics
-- Uses fact_sales_enriched which already has holiday columns
CREATE OR REPLACE TABLE workspace.gold.metrics_holiday_performance AS
WITH holiday_comparison AS (
  SELECT
    CASE WHEN is_holiday THEN 'Holiday' ELSE 'Non-Holiday' END AS day_type,
    COUNT(DISTINCT order_date) AS unique_days,
    COUNT(DISTINCT order_number) AS order_count,
    SUM(sales_amount) AS total_revenue,
    SUM(quantity) AS total_items,
    AVG(sales_amount) AS avg_order_value,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(sales_amount) / COUNT(DISTINCT order_date) AS avg_revenue_per_day
  FROM workspace.gold.fact_sales_enriched
  GROUP BY CASE WHEN is_holiday THEN 'Holiday' ELSE 'Non-Holiday' END
),
holiday_rankings AS (
  SELECT
    holiday_name,
    COUNT(DISTINCT order_number) AS order_count,
    SUM(sales_amount) AS total_revenue,
    AVG(sales_amount) AS avg_order_value,
    SUM(quantity) AS total_items,
    RANK() OVER (ORDER BY SUM(sales_amount) DESC) AS revenue_rank
  FROM workspace.gold.fact_sales_enriched
  WHERE is_holiday = TRUE
  GROUP BY holiday_name
),
pre_post_analysis AS (
  SELECT
    holiday_name,
    CASE 
      WHEN is_pre_holiday_week THEN 'Pre-Holiday (7 days)'
      WHEN is_holiday THEN 'Holiday'
      WHEN is_post_holiday_week THEN 'Post-Holiday (7 days)'
      ELSE 'Regular'
    END AS period_type,
    SUM(sales_amount) AS total_revenue,
    COUNT(DISTINCT order_number) AS order_count,
    AVG(sales_amount) AS avg_order_value
  FROM workspace.gold.fact_sales_enriched
  WHERE holiday_name IS NOT NULL OR (is_pre_holiday_week OR is_post_holiday_week)
  GROUP BY holiday_name, 
    CASE 
      WHEN is_pre_holiday_week THEN 'Pre-Holiday (7 days)'
      WHEN is_holiday THEN 'Holiday'
      WHEN is_post_holiday_week THEN 'Post-Holiday (7 days)'
      ELSE 'Regular'
    END
)
SELECT
  'Holiday Comparison' AS metric_category,
  day_type,
  unique_days,
  order_count,
  total_revenue,
  avg_revenue_per_day,
  avg_order_value,
  unique_customers,
  CAST(NULL AS STRING) AS holiday_name,
  CAST(NULL AS INT) AS revenue_rank,
  CAST(NULL AS STRING) AS period_type
FROM holiday_comparison

UNION ALL

SELECT
  'Holiday Rankings' AS metric_category,
  CAST(NULL AS STRING) AS day_type,
  CAST(NULL AS BIGINT) AS unique_days,
  order_count,
  total_revenue,
  CAST(NULL AS DOUBLE) AS avg_revenue_per_day,
  avg_order_value,
  CAST(NULL AS BIGINT) AS unique_customers,
  holiday_name,
  revenue_rank,
  CAST(NULL AS STRING) AS period_type
FROM holiday_rankings

UNION ALL

SELECT
  'Pre/Post Holiday Trends' AS metric_category,
  CAST(NULL AS STRING) AS day_type,
  CAST(NULL AS BIGINT) AS unique_days,
  order_count,
  total_revenue,
  CAST(NULL AS DOUBLE) AS avg_revenue_per_day,
  avg_order_value,
  CAST(NULL AS BIGINT) AS unique_customers,
  holiday_name,
  CAST(NULL AS INT) AS revenue_rank,
  period_type
FROM pre_post_analysis;

-- Calculate and display holiday lift (CORRECTED: uses per-day averages)
WITH holiday_lift AS (
  SELECT
    SUM(CASE WHEN metric_category = 'Holiday Comparison' AND day_type = 'Holiday' THEN avg_revenue_per_day END) AS holiday_avg_per_day,
    SUM(CASE WHEN metric_category = 'Holiday Comparison' AND day_type = 'Non-Holiday' THEN avg_revenue_per_day END) AS non_holiday_avg_per_day
  FROM workspace.gold.metrics_holiday_performance
)
SELECT 
  holiday_avg_per_day,
  non_holiday_avg_per_day,
  ROUND((holiday_avg_per_day / NULLIF(non_holiday_avg_per_day, 0) - 1) * 100, 2) AS holiday_lift_pct
FROM holiday_lift;

## 2️⃣ Customer Behavior Metrics

**Business Value**:
- Identify high-value customer cohorts for targeted retention
- Measure repeat purchase behavior to optimize loyalty programs
- Understand churn patterns to reduce customer attrition
- Segment customers by RFM for personalized marketing

**Metrics Included**:
- Customer Lifetime Value (CLV) by cohort
- Repeat purchase rate
- Customer retention & churn by cohort
- RFM segmentation (Recency, Frequency, Monetary)

In [0]:
%sql
-- Create comprehensive customer behavior metrics
CREATE OR REPLACE TABLE workspace.gold.metrics_customer_behavior AS
WITH customer_orders AS (
  SELECT
    f.customer_id,
    c.first_name,
    c.last_name,
    c.gender,
    c.marital_status,
    DATE_TRUNC('month', MIN(f.order_date)) AS first_order_month,
    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.sales_amount) AS lifetime_value,
    AVG(f.sales_amount) AS avg_order_value,
    MIN(f.order_date) AS first_order_date,
    MAX(f.order_date) AS last_order_date,
    DATEDIFF(MAX(f.order_date), MIN(f.order_date)) AS customer_lifetime_days
  FROM workspace.gold.fact_sales_enriched f
  INNER JOIN workspace.gold.dim_customers c ON f.customer_id = c.customer_id
  GROUP BY f.customer_id, c.first_name, c.last_name, c.gender, c.marital_status
),
repeat_customers AS (
  SELECT
    first_order_month,
    COUNT(CASE WHEN total_orders > 1 THEN 1 END) AS repeat_customers,
    COUNT(*) AS total_customers,
    ROUND(COUNT(CASE WHEN total_orders > 1 THEN 1 END) * 100.0 / COUNT(*), 2) AS repeat_purchase_rate_pct
  FROM customer_orders
  GROUP BY first_order_month
),
rfm_segmentation AS (
  SELECT
    customer_id,
    first_name,
    last_name,
    DATEDIFF(CURRENT_DATE(), last_order_date) AS recency_days,
    total_orders AS frequency,
    lifetime_value AS monetary,
    NTILE(5) OVER (ORDER BY DATEDIFF(CURRENT_DATE(), last_order_date) DESC) AS recency_score,
    NTILE(5) OVER (ORDER BY total_orders ASC) AS frequency_score,
    NTILE(5) OVER (ORDER BY lifetime_value ASC) AS monetary_score
  FROM customer_orders
),
rfm_classification AS (
  SELECT
    *,
    CASE
      WHEN recency_score >= 4 AND frequency_score >= 4 AND monetary_score >= 4 THEN 'Champions'
      WHEN recency_score >= 3 AND frequency_score >= 3 THEN 'Loyal Customers'
      WHEN recency_score >= 4 AND frequency_score <= 2 THEN 'Promising'
      WHEN recency_score <= 2 AND frequency_score >= 3 THEN 'At Risk'
      WHEN recency_score <= 2 AND frequency_score <= 2 THEN 'Lost'
      ELSE 'Others'
    END AS customer_segment
  FROM rfm_segmentation
)
SELECT
  'Customer Cohort Summary' AS metric_category,
  co.first_order_month AS cohort_month,
  COUNT(*) AS cohort_size,
  SUM(co.lifetime_value) AS cohort_total_clv,
  AVG(co.lifetime_value) AS cohort_avg_clv,
  AVG(co.total_orders) AS cohort_avg_orders,
  MAX(rc.repeat_purchase_rate_pct) AS repeat_purchase_rate_pct,
  CAST(NULL AS STRING) AS customer_segment
FROM customer_orders co
LEFT JOIN repeat_customers rc ON co.first_order_month = rc.first_order_month
GROUP BY co.first_order_month

UNION ALL

SELECT
  'RFM Segmentation' AS metric_category,
  CAST(NULL AS TIMESTAMP) AS cohort_month,
  COUNT(*) AS cohort_size,
  SUM(monetary) AS cohort_total_clv,
  AVG(monetary) AS cohort_avg_clv,
  AVG(frequency) AS cohort_avg_orders,
  CAST(NULL AS DOUBLE) AS repeat_purchase_rate_pct,
  customer_segment
FROM rfm_classification
GROUP BY customer_segment;

## 3️⃣ Product Performance Metrics

**Business Value**:
- Identify top revenue-generating products for strategic focus
- Discover product affinity patterns for cross-sell opportunities
- Compare holiday vs non-holiday performance for seasonal planning
- Classify inventory velocity to optimize stock levels

**Metrics Included**:
- Top Products by Revenue & Quantity
- Product Affinity (Basket Analysis)
- Holiday vs Regular Day Performance
- Product Velocity Classification (Fast/Slow Moving)

In [0]:
%sql
-- Drop old wide table
DROP TABLE IF EXISTS workspace.gold.metrics_product_performance;

-- Create intermediate tables (not temp views - free edition doesn't support them)
CREATE OR REPLACE TABLE workspace.default.product_sales AS
  SELECT
    f.product_key,
    f.order_number,
    f.order_date,
    f.sales_amount,
    f.quantity,
    f.price,
    f.is_holiday,
    f.holiday_name,
    CASE WHEN f.is_holiday THEN 'Holiday' ELSE 'Regular' END AS day_type
  FROM workspace.gold.fact_sales_enriched f;

CREATE OR REPLACE TABLE workspace.default.top_products AS
  SELECT
    product_key,
    SUM(sales_amount) AS total_revenue,
    SUM(quantity) AS total_quantity,
    COUNT(DISTINCT order_number) AS order_count,
    AVG(price) AS avg_price,
    RANK() OVER (ORDER BY SUM(sales_amount) DESC) AS revenue_rank
  FROM workspace.default.product_sales
  GROUP BY product_key;

CREATE OR REPLACE TABLE workspace.default.product_affinity AS
  SELECT
    a.product_key AS product_a,
    b.product_key AS product_b,
    COUNT(DISTINCT a.order_number) AS co_occurrence_count,
    RANK() OVER (PARTITION BY a.product_key ORDER BY COUNT(DISTINCT a.order_number) DESC) AS affinity_rank
  FROM workspace.default.product_sales a
  INNER JOIN workspace.default.product_sales b 
    ON a.order_number = b.order_number 
    AND a.product_key < b.product_key
  GROUP BY a.product_key, b.product_key;

CREATE OR REPLACE TABLE workspace.default.holiday_vs_regular AS
  SELECT
    product_key,
    day_type,
    SUM(sales_amount) AS total_revenue,
    COUNT(DISTINCT order_number) AS order_count,
    AVG(sales_amount) AS avg_order_value
  FROM workspace.default.product_sales
  GROUP BY product_key, day_type;

CREATE OR REPLACE TABLE workspace.default.product_velocity AS
  SELECT
    product_key,
    COUNT(DISTINCT order_number) AS order_frequency,
    SUM(quantity) AS total_quantity,
    CASE
      WHEN COUNT(DISTINCT order_number) > 100 THEN 'Fast Moving'
      WHEN COUNT(DISTINCT order_number) BETWEEN 50 AND 100 THEN 'Medium Moving'
      ELSE 'Slow Moving'
    END AS velocity_classification
  FROM workspace.default.product_sales
  GROUP BY product_key;

-- Table 1: Top 50 Products by Revenue
CREATE OR REPLACE TABLE workspace.gold.metrics_product_top_performers AS
SELECT
  product_key,
  total_revenue,
  total_quantity,
  order_count,
  avg_price,
  revenue_rank
FROM workspace.default.top_products
WHERE revenue_rank <= 50;

-- Table 2: Product Affinity (Top 5 per product)
CREATE OR REPLACE TABLE workspace.gold.metrics_product_affinity AS
SELECT
  product_a,
  product_b,
  co_occurrence_count,
  affinity_rank
FROM workspace.default.product_affinity
WHERE affinity_rank <= 5;

-- Table 3: Holiday vs Regular Performance
CREATE OR REPLACE TABLE workspace.gold.metrics_product_holiday_performance AS
SELECT
  product_key,
  day_type,
  total_revenue,
  order_count,
  avg_order_value
FROM workspace.default.holiday_vs_regular;

-- Table 4: Product Velocity Classification
CREATE OR REPLACE TABLE workspace.gold.metrics_product_velocity AS
SELECT
  product_key,
  order_frequency,
  total_quantity,
  velocity_classification
FROM workspace.default.product_velocity;

SELECT 'Product metrics refactored into 4 focused tables' AS status;

## 4️⃣ Time-based Metrics

**Business Value**:
- Identify peak sales days for staffing optimization
- Track growth trends to measure business health
- Understand seasonality for demand forecasting
- Compare year-over-year performance

**Metrics Included**:
- Sales by Day of Week
- Month-over-Month Growth Rate
- Seasonal Patterns (by Quarter)
- Year-over-Year Comparisons

In [0]:
%sql
-- Drop old wide table
DROP TABLE IF EXISTS workspace.gold.metrics_time_trends;

-- Create intermediate tables (not temp views - free edition doesn't support them)
CREATE OR REPLACE TABLE workspace.default.daily_sales AS
  SELECT
    order_date,
    DAYOFWEEK(order_date) AS day_of_week,
    DAYNAME(order_date) AS day_name,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    QUARTER(order_date) AS quarter,
    DATE_TRUNC('month', order_date) AS month_start,
    SUM(sales_amount) AS daily_revenue,
    COUNT(DISTINCT order_number) AS daily_orders,
    SUM(quantity) AS daily_items
  FROM workspace.gold.fact_sales_enriched
  GROUP BY order_date;

CREATE OR REPLACE TABLE workspace.default.day_of_week_summary AS
  SELECT
    day_of_week,
    day_name,
    SUM(daily_revenue) AS total_revenue,
    AVG(daily_revenue) AS avg_daily_revenue,
    SUM(daily_orders) AS total_orders,
    AVG(daily_orders) AS avg_daily_orders
  FROM workspace.default.daily_sales
  GROUP BY day_of_week, day_name;

CREATE OR REPLACE TABLE workspace.default.monthly_trends AS
  SELECT
    month_start,
    SUM(daily_revenue) AS monthly_revenue,
    SUM(daily_orders) AS monthly_orders,
    LAG(SUM(daily_revenue)) OVER (ORDER BY month_start) AS prev_month_revenue,
    (SUM(daily_revenue) - LAG(SUM(daily_revenue)) OVER (ORDER BY month_start)) / 
      NULLIF(LAG(SUM(daily_revenue)) OVER (ORDER BY month_start), 0) * 100 AS mom_growth_pct
  FROM workspace.default.daily_sales
  GROUP BY month_start;

CREATE OR REPLACE TABLE workspace.default.seasonal_patterns AS
  SELECT
    year,
    quarter,
    SUM(daily_revenue) AS quarterly_revenue,
    AVG(daily_revenue) AS avg_daily_revenue,
    SUM(daily_orders) AS quarterly_orders
  FROM workspace.default.daily_sales
  GROUP BY year, quarter;

CREATE OR REPLACE TABLE workspace.default.yoy_comparison AS
  SELECT
    year,
    month,
    SUM(daily_revenue) AS monthly_revenue,
    LAG(SUM(daily_revenue), 12) OVER (ORDER BY year, month) AS prev_year_revenue,
    (SUM(daily_revenue) - LAG(SUM(daily_revenue), 12) OVER (ORDER BY year, month)) / 
      NULLIF(LAG(SUM(daily_revenue), 12) OVER (ORDER BY year, month), 0) * 100 AS yoy_growth_pct
  FROM workspace.default.daily_sales
  GROUP BY year, month;

-- Table 1: Day of Week Aggregates
CREATE OR REPLACE TABLE workspace.gold.metrics_time_day_of_week AS
SELECT
  day_of_week,
  day_name,
  total_revenue,
  avg_daily_revenue,
  total_orders,
  avg_daily_orders
FROM workspace.default.day_of_week_summary;

-- Table 2: Month-over-Month Trends
CREATE OR REPLACE TABLE workspace.gold.metrics_time_monthly_trends AS
SELECT
  DATE_FORMAT(month_start, 'yyyy-MM') AS month_period,
  month_start,
  monthly_revenue,
  monthly_orders,
  prev_month_revenue,
  mom_growth_pct
FROM workspace.default.monthly_trends;

-- Table 3: Seasonal Patterns (Quarterly)
CREATE OR REPLACE TABLE workspace.gold.metrics_time_seasonal_patterns AS
SELECT
  year,
  quarter,
  CONCAT(year, '-Q', quarter) AS quarter_period,
  quarterly_revenue,
  avg_daily_revenue,
  quarterly_orders
FROM workspace.default.seasonal_patterns;

-- Table 4: Year-over-Year Comparison
CREATE OR REPLACE TABLE workspace.gold.metrics_time_yoy_comparison AS
SELECT
  year,
  month,
  CONCAT(year, '-', LPAD(month, 2, '0')) AS month_period,
  monthly_revenue,
  prev_year_revenue,
  yoy_growth_pct
FROM workspace.default.yoy_comparison
WHERE prev_year_revenue IS NOT NULL;

SELECT 'Time metrics refactored into 4 focused tables' AS status;

## 5️⃣ Operational Metrics

**Business Value**:
- Monitor fulfillment efficiency to improve customer satisfaction
- Identify late shipments to address operational bottlenecks
- Track order backlog for capacity planning
- Measure operational performance KPIs

**Metrics Included**:
- Average Fulfillment Time (order to ship)
- On-time Delivery Rate
- Late Shipment Analysis
- Order Backlog by Status

In [0]:
%sql
-- Create comprehensive operational metrics
CREATE OR REPLACE TABLE workspace.gold.metrics_operations AS
WITH order_fulfillment AS (
  SELECT
    order_number,
    product_key,
    customer_id,
    order_date,
    ship_date,
    due_date,
    sales_amount,
    quantity,
    DATEDIFF(ship_date, order_date) AS fulfillment_time_days,
    DATEDIFF(due_date, ship_date) AS time_to_due_days,
    CASE WHEN ship_date <= due_date THEN 'On Time' ELSE 'Late' END AS delivery_status
  FROM workspace.gold.fact_sales_enriched
),
fulfillment_summary AS (
  SELECT
    AVG(fulfillment_time_days) AS avg_fulfillment_time,
    PERCENTILE(fulfillment_time_days, 0.5) AS median_fulfillment_time,
    MAX(fulfillment_time_days) AS max_fulfillment_time,
    MIN(fulfillment_time_days) AS min_fulfillment_time
  FROM order_fulfillment
  WHERE fulfillment_time_days >= 0
),
delivery_performance AS (
  SELECT
    delivery_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
  FROM order_fulfillment
  GROUP BY delivery_status
),
backlog_analysis AS (
  SELECT
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    COUNT(CASE WHEN delivery_status = 'Late' THEN 1 END) AS late_orders,
    COUNT(*) AS total_orders,
    ROUND(COUNT(CASE WHEN delivery_status = 'Late' THEN 1 END) * 100.0 / COUNT(*), 2) AS late_order_pct
  FROM order_fulfillment
  GROUP BY YEAR(order_date), MONTH(order_date)
)
SELECT
  'Fulfillment Time' AS metric_category,
  CAST(NULL AS STRING) AS dimension,
  avg_fulfillment_time AS metric_value,
  median_fulfillment_time AS secondary_value,
  CAST(NULL AS DOUBLE) AS percentage
FROM fulfillment_summary

UNION ALL

SELECT
  'Delivery Performance' AS metric_category,
  delivery_status AS dimension,
  order_count AS metric_value,
  CAST(NULL AS DOUBLE) AS secondary_value,
  percentage
FROM delivery_performance

UNION ALL

SELECT
  'Monthly Backlog' AS metric_category,
  CONCAT(year, '-', LPAD(month, 2, '0')) AS dimension,
  late_orders AS metric_value,
  total_orders AS secondary_value,
  late_order_pct AS percentage
FROM backlog_analysis;

## 6️⃣ Executive Summary Dashboard

**Business Value**:
- Single view of all key business metrics for executive reporting
- Quick snapshot of business health across all dimensions
- Supports data-driven decision making at the leadership level

**Metrics Included**:
- Top-line revenue & growth
- Customer metrics (CLV, retention, segments)
- Product performance highlights
- Operational efficiency KPIs
- Holiday impact summary

In [0]:
%sql
-- Create executive summary combining all key metrics
CREATE OR REPLACE TABLE workspace.gold.metrics_executive_summary AS
WITH revenue_summary AS (
  SELECT
    SUM(sales_amount) AS total_revenue,
    COUNT(DISTINCT order_number) AS total_orders,
    AVG(sales_amount) AS avg_order_value,
    COUNT(DISTINCT customer_id) AS unique_customers
  FROM workspace.gold.fact_sales_enriched
),
holiday_impact AS (
  SELECT
    SUM(CASE WHEN is_holiday THEN sales_amount ELSE 0 END) AS holiday_revenue,
    SUM(CASE WHEN NOT is_holiday THEN sales_amount ELSE 0 END) AS non_holiday_revenue,
    ROUND((SUM(CASE WHEN is_holiday THEN sales_amount ELSE 0 END) / 
           NULLIF(SUM(CASE WHEN NOT is_holiday THEN sales_amount ELSE 0 END), 0) - 1) * 100, 2) AS holiday_lift_pct
  FROM workspace.gold.fact_sales_enriched
),
top_products AS (
  SELECT
    product_key,
    SUM(sales_amount) AS total_revenue,
    RANK() OVER (ORDER BY SUM(sales_amount) DESC) AS revenue_rank
  FROM workspace.gold.fact_sales_enriched
  GROUP BY product_key
),
operational_metrics AS (
  SELECT
    AVG(DATEDIFF(ship_date, order_date)) AS avg_fulfillment_days,
    ROUND(SUM(CASE WHEN ship_date <= due_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS on_time_pct
  FROM workspace.gold.fact_sales_enriched
)
SELECT
  'Revenue Metrics' AS metric_category,
  'Total Revenue' AS metric_name,
  total_revenue AS metric_value,
  '$' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM revenue_summary

UNION ALL

SELECT
  'Revenue Metrics' AS metric_category,
  'Total Orders' AS metric_name,
  total_orders AS metric_value,
  'orders' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM revenue_summary

UNION ALL

SELECT
  'Revenue Metrics' AS metric_category,
  'Avg Order Value' AS metric_name,
  avg_order_value AS metric_value,
  '$' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM revenue_summary

UNION ALL

SELECT
  'Customer Metrics' AS metric_category,
  'Unique Customers' AS metric_name,
  unique_customers AS metric_value,
  'customers' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM revenue_summary

UNION ALL

SELECT
  'Holiday Impact' AS metric_category,
  'Holiday Lift %' AS metric_name,
  holiday_lift_pct AS metric_value,
  '%' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM holiday_impact

UNION ALL

SELECT
  'Top Products' AS metric_category,
  product_key AS metric_name,
  total_revenue AS metric_value,
  '$' AS unit,
  revenue_rank AS rank_position
FROM top_products
WHERE revenue_rank <= 5

UNION ALL

SELECT
  'Operations' AS metric_category,
  'Avg Fulfillment Days' AS metric_name,
  avg_fulfillment_days AS metric_value,
  'days' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM operational_metrics

UNION ALL

SELECT
  'Operations' AS metric_category,
  'On-Time Delivery %' AS metric_name,
  on_time_pct AS metric_value,
  '%' AS unit,
  CAST(NULL AS INT) AS rank_position
FROM operational_metrics;

## ✅ Metrics Mart Complete

### Created Tables:

1. **`workspace.gold.metrics_holiday_performance`** - Holiday sales analysis with lift calculations
2. **`workspace.gold.metrics_customer_behavior`** - CLV, RFM, cohort, and churn metrics
3. **`workspace.gold.metrics_product_performance`** - Product rankings, affinity, and velocity
4. **`workspace.gold.metrics_time_trends`** - Day of week, MoM/YoY growth, seasonality
5. **`workspace.gold.metrics_operations`** - Fulfillment, delivery, and backlog KPIs
6. **`workspace.gold.metrics_executive_summary`** - Single-view executive dashboard

🔄 **Refresh Strategy**: Schedule this notebook to run when upstream data updates (see below)

---

### 🚀 Next Steps for Analytics Engineers:

**1. Schedule Automatic Refresh:**

Create a Databricks Job to run this notebook when `fact_sales_enriched` updates:

```python
# Option A: Daily refresh at 8 AM
# Schedule: Daily at 8:00 AM
# Cron: 0 0 8 * * ? *

# Option B: Hourly refresh
# Schedule: Every 1 hour
# Cron: 0 0 */1 * * ? *

# Option C: After upstream pipeline completes
# Add as downstream task after fact_sales_enriched pipeline
```

**2. Add Incremental Logic (Optional Optimization):**

For large datasets, consider incremental refresh:
```sql
-- Only process recent data
WHERE order_date >= DATE_SUB(CURRENT_DATE(), 7)
```

**3. Connect BI Tools:**
* Point Tableau/Power BI/Looker to `workspace.gold.metrics_*` tables
* Fast queries (pre-computed aggregations)
* Single source of truth for all business metrics

**4. Monitor & Optimize:**
* Add data quality checks
* Set up alerts for refresh failures
* Monitor query performance with Databricks SQL Query History

---

### 📊 Sample Queries:

```sql
-- Top holidays by revenue
SELECT holiday_name, total_revenue 
FROM workspace.gold.metrics_holiday_performance
WHERE metric_category = 'Holiday Rankings'
ORDER BY revenue_rank LIMIT 10;

-- Champion customers
SELECT customer_segment, cohort_size, cohort_avg_clv
FROM workspace.gold.metrics_customer_behavior
WHERE metric_category = 'RFM Segmentation' 
  AND customer_segment = 'Champions';

-- Executive KPIs
SELECT metric_category, metric_name, metric_value, unit
FROM workspace.gold.metrics_executive_summary
ORDER BY metric_category, metric_name;

-- Month-over-month growth trends
SELECT time_period, total_revenue, growth_pct
FROM workspace.gold.metrics_time_trends
WHERE metric_category = 'Month over Month'
ORDER BY time_period DESC LIMIT 12;
```

---

### ⬆️ Future Upgrade Path:

**When Materialized Views become available on Serverless:**

1. Convert tables to materialized views:
```sql
DROP TABLE workspace.gold.metrics_holiday_performance;
CREATE MATERIALIZED VIEW workspace.gold.metrics_holiday_performance
TRIGGER ON UPDATE
AS
-- (same query logic)
```

2. Benefits:
* Automatic refresh when upstream data changes
* No job scheduling needed
* Predictive Optimization handles all maintenance
* Lower operational overhead